# Stage 2 follow-up: Degraded Sessions
## ISM 6562 — Final Project — MediStream Telehealth
---
Closes a gap in `02-spark-transforms.ipynb`. The brief's Stage 2 guide calls for a feature that flags sessions where quality dropped below thresholds during the appointment. This is the **batch/historical** view of session degradation — Stage 3 (Kafka + Spark Structured Streaming) will compute the real-time, mid-call version.

Because the curated `session_quality` table holds one aggregate row per session (not the per-sample stream that Stage 3 will consume), this batch detector works off summary metrics rather than mid-call sliding-window dips. It uses the same thresholds the streaming alerts will use, so the two views agree on what "degraded" means.

**Inputs:**
- `/medistream/curated/session_quality`
- `/medistream/curated/appointments`
- `/medistream/curated/patient_feedback`

**Output:** `/medistream/analytics/degraded_sessions` (Parquet, partitioned by `degraded_severity`)

**Output schema:** `session_id, appointment_id, physician_id, specialty, visit_type, latency_ms, packet_loss_pct, audio_quality_score, video_resolution, device_type, os, degraded_latency, degraded_packet_loss, degraded_audio, degraded_severity, nps_score, rating`

## Setup

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName('MediStream-Stage2e-DegradedSessions')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate())

HDFS_CURATED   = 'hdfs://namenode:9000/medistream/curated'
HDFS_ANALYTICS = 'hdfs://namenode:9000/medistream/analytics'
LOCAL_CURATED   = '/home/jovyan/data/curated'
LOCAL_ANALYTICS = '/home/jovyan/data/analytics'

try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ANALYTICS}/_probe')
    CURATED, ANALYTICS = HDFS_CURATED, HDFS_ANALYTICS
    print('Using HDFS')
except Exception:
    CURATED, ANALYTICS = LOCAL_CURATED, LOCAL_ANALYTICS
    print('HDFS unavailable, using local mount')

## Build the enriched session view

Join session_quality with appointments to get physician + specialty context, then left-join feedback to attach satisfaction signal where available.

In [ ]:
session  = spark.read.parquet(f'{CURATED}/session_quality')
appts    = spark.read.parquet(f'{CURATED}/appointments')
feedback = spark.read.parquet(f'{CURATED}/patient_feedback')

appts_lite = appts.select('appointment_id', 'physician_id', 'specialty', 'visit_type')
feedback_lite = feedback.select(
    'appointment_id',
    F.col('rating').alias('rating'),
    F.col('nps_score').alias('nps_score'),
)

enriched = (session
    .join(appts_lite, on='appointment_id', how='left')
    .join(feedback_lite, on='appointment_id', how='left')
)
print(f'Enriched session rows: {enriched.count():,}')

## Apply degradation flags

Three independent flags, summed into a `degraded_severity` score (0–3) used as the partition key.

| Flag | Trigger |
|---|---|
| `degraded_latency`     | `latency_ms > 500` (matches Stage 3 alert threshold) |
| `degraded_packet_loss` | `packet_loss_pct > 5` (matches Stage 3 alert threshold) |
| `degraded_audio`       | `audio_quality_score < 5` (mid-scale on the codec's 1–10 score) |

In [ ]:
LATENCY_THRESHOLD = 500
PACKET_LOSS_THRESHOLD = 5.0
LOW_AUDIO_THRESHOLD = 5

flagged = (enriched
    .withColumn('degraded_latency',
        (F.col('latency_ms') > LATENCY_THRESHOLD).cast('int'))
    .withColumn('degraded_packet_loss',
        (F.col('packet_loss_pct') > PACKET_LOSS_THRESHOLD).cast('int'))
    .withColumn('degraded_audio',
        (F.col('audio_quality_score') < LOW_AUDIO_THRESHOLD).cast('int'))
    .withColumn('degraded_severity',
        F.col('degraded_latency') + F.col('degraded_packet_loss') + F.col('degraded_audio'))
)

print('Severity distribution:')
flagged.groupBy('degraded_severity').count().orderBy('degraded_severity').show()

## Write partitioned by severity

Partitioning by `degraded_severity` lets dashboards query "show me all sessions with severity ≥ 2" by reading just two partitions instead of scanning the full table.

In [ ]:
out_path = f'{ANALYTICS}/degraded_sessions'
(flagged.write
    .mode('overwrite')
    .partitionBy('degraded_severity')
    .parquet(out_path))

print(f'Wrote {flagged.count():,} session rows to {out_path}')

## Verify and inspect

Quick correlation check — does NPS drop on degraded sessions? The brief reports NPS drops from 72 (smooth) to 31 (with quality issues). This table lets us reproduce that finding from data.

In [ ]:
verify = spark.read.parquet(f'{ANALYTICS}/degraded_sessions')
print(f'Read back {verify.count():,} rows, {len(verify.columns)} columns')
verify.printSchema()

print('\nAvg NPS by degradation severity (only rows with feedback):')
(verify.filter(F.col('nps_score').isNotNull())
    .groupBy('degraded_severity')
    .agg(
        F.count('*').alias('sessions_with_feedback'),
        F.round(F.avg('nps_score'), 2).alias('avg_nps'),
    )
    .orderBy('degraded_severity')
    .show())

print('\nSample degraded sessions (severity 3):')
verify.filter(F.col('degraded_severity') == 3).show(10, truncate=False)